In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/non_exploratory/check'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/noise_model_sanity_check_paths.json')


# Sanity Check for the Noise Model

MPP == Z meas ?

In [ ]:
import multiprocessing
import sys
from pathlib import Path
import pandas as pd
import sinter
import matplotlib.pyplot as plt
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders.cpp_backend.bp_osd import BPOSD
from src.decoders.cpp_backend.tesseract import Tesseract

DATA_DIR = _data_paths.stats()
DATA_DIR.mkdir(parents=True, exist_ok=True)
REF_DIR = _data_paths.input('reference')
%load_ext autoreload
%autoreload 2


In [ ]:
sinter.plot_error_rate


In [ ]:
data_bposd_file = _data_paths.input('2026_04_14_data_err_rates_c_ldpc_dec_bposd_dist_5_r_3_shots_1_0e_09_txt')
data_tesseract_file = _data_paths.input('2026_04_15_data_err_rates_c_ldpc_dec_tesseract_dist_5_r_3_shots_1_0e_09_txt')
data_bposd = pd.read_csv(data_bposd_file, comment='#', sep='\\s+', header=None)

data_bposd.columns = ['p_noise', 'p_best', 'p_low', 'p_high']
data_tesseract = pd.read_csv(data_tesseract_file, comment='#', sep='\\s+', header=None)

data_tesseract.columns = ['p_noise', 'p_best', 'p_low', 'p_high']


In [ ]:
data_bposd


In [ ]:
data_tesseract


In [ ]:
ca_code = CA_9_2_5()
p_values_bposd = data_bposd['p_noise'].values
p_values_tesseract = data_tesseract['p_noise'].values
max_shots = 1000000000
max_errors = 2000
save_file = _data_paths.output('bposd_tesseract_d_5_r_3_csv')


In [ ]:
tasks = []
custom_decoders = {}
builder = CircuitBuilder(ca_code)

for p in p_values_bposd:
    circuit = builder.get_circuit_level_circuit(error_rate=ErrorRate(p=float(p)), d_rounds=3)
    bposd = bposd = BPOSD(
        code=ca_code,
        error_rate=float(p),
        max_iter=60,
        bp_method='minimum_sum',
        osd_order=10,
        osd_method='osd_cs',
    )
    b_key = f'BPOSD_p_{p:.6g}'
    custom_decoders[b_key] = bposd
    tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={'code': 'CA_9_2_5', 'decoder': 'BPOSD', 'p': float(p)}))

for p in p_values_tesseract:
    circuit = builder.get_circuit_level_circuit(error_rate=ErrorRate(p=float(p)), d_rounds=3)
    tess = Tesseract(
        code=ca_code,
        error_rate=float(p),
        beam=8,
        pq_limit=200000,
        det_penalty=0.0,
    )
    t_key = f'Tesseract_p_{p:.6g}'
    custom_decoders[t_key] = tess
    tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={'code': 'CA_9_2_5', 'decoder': 'Tesseract', 'p': float(p)}))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=True,
    save_resume_filepath=save_file,
)


In [ ]:
def parse_stats(stats):
    rows = []
    for stat in stats:
        shots = stat.shots
        errors = stat.errors
        p_noise = stat.json_metadata['p']
        p_fit = sinter.fit_binomial(num_shots=shots, num_hits=errors, max_likelihood_factor=1000.0)
        rows.append({
            'p_noise': p_noise,
            'p_best': p_fit.best,
            'p_low': p_fit.low,
            'p_high': p_fit.high,
        })

    return pd.DataFrame(rows)


In [ ]:
stats = sinter.read_stats_from_csv_files(save_file)
bposd_stats = [stat for stat in stats if stat.json_metadata['decoder'] == 'BPOSD']
tesseract_stats = [stat for stat in stats if stat.json_metadata['decoder'] == 'Tesseract']


In [ ]:
bposd_parsed = parse_stats(bposd_stats)
bposd_parsed


In [ ]:
tesseract_parsed = parse_stats(tesseract_stats)
tesseract_parsed


In [ ]:
plt.errorbar(data_bposd['p_noise'], data_bposd['p_best'], yerr=[data_bposd['p_best'] - data_bposd['p_low'], data_bposd['p_high'] - data_bposd['p_best']], fmt='o', label='reference')
plt.errorbar(bposd_parsed['p_noise'], bposd_parsed['p_best'], yerr=[bposd_parsed['p_best'] - bposd_parsed['p_low'], bposd_parsed['p_high'] - bposd_parsed['p_best']], fmt='x', label='mine')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Physical error rate')
plt.ylabel('Logical error rate')
plt.legend()

plt.show()


In [ ]:
plt.errorbar(data_tesseract['p_noise'], data_tesseract['p_best'], yerr=[data_tesseract['p_best'] - data_tesseract['p_low'], data_tesseract['p_high'] - data_tesseract['p_best']], fmt='o', label='reference')
plt.errorbar(tesseract_parsed['p_noise'], tesseract_parsed['p_best'], yerr=[tesseract_parsed['p_best'] - tesseract_parsed['p_low'], tesseract_parsed['p_high'] - tesseract_parsed['p_best']], fmt='x', label='mine')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Physical error rate')
plt.ylabel('Logical error rate')
plt.legend()

plt.show()
